# Plot spot position as time series data
env : data_vis_32

# 1.0 Import relevant packages

In [2]:
# import pypyodbc
import pandas as pd
import plotly.express as px
import seaborn as sns
from matplotlib.colors import to_hex

import re
from pathlib import Path

# 2.0 Import spot position and size QA data

In [3]:
data_path = r"../data/xlsx_exported_from_access/SpotPositionResults.xlsx"

df = pd.read_excel(data_path)

df.head(2)



,ADate,MachineName,Energy,Device,Gantry Angle,Spot,x-pos,y-pos,hor_rt_gradient,hor_lt_gradient,hor_fwhm,vert_rt_gradient,vert_lt_gradient,vert_fwhm,bltr_rt_gradient,bltr_lt_gradient,bltr_fwhm,tlbr_rt_gradient,tlbr_lt_gradient,tlbr_fwhm
0,2022-04-21 16:44:14,Gantry 2,70,XRV-3000,180,Bottom-Centre,-0.2357,124.9224,-9.059840,9.157258,13.342140,-8.964427,9.333333,13.536155,-8.485281,8.747554,14.216962,-8.909545,8.992812,13.842831
1,2022-04-21 16:44:14,Gantry 2,70,XRV-3000,180,Bottom-Left,-125.0153,125.4501,-8.871094,9.120482,13.562307,-8.964427,9.333333,13.712522,-8.747554,8.591347,14.310494,-8.747554,8.992812,13.842831


# 3.0 exploratory data analysis - understand your data

In [4]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41172 entries, 0 to 41171
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   ADate             41172 non-null  datetime64[ns]
 1   MachineName       41172 non-null  object        
 2   Energy            41172 non-null  int64         
 3   Device            41172 non-null  object        
 4   Gantry Angle      41172 non-null  int64         
 5   Spot              41172 non-null  object        
 6   x-pos             41172 non-null  float64       
 7   y-pos             41172 non-null  float64       
 8   hor_rt_gradient   41172 non-null  float64       
 9   hor_lt_gradient   41172 non-null  float64       
 10  hor_fwhm          41172 non-null  float64       
 11  vert_rt_gradient  41172 non-null  float64       
 12  vert_lt_gradient  41172 non-null  float64       
 13  vert_fwhm         41172 non-null  float64       
 14  bltr_rt_gradient  4117

In [5]:
df.value_counts("MachineName"), df.value_counts("Device"), df.value_counts("Energy")

(MachineName
 Gantry 3    10576
 Gantry 1    10472
 Gantry 4    10357
 Gantry 2     9767
 Name: count, dtype: int64,
 Device
 XRV-3000    31815
 XRV-4000     9357
 Name: count, dtype: int64,
 Energy
 150    8250
 240    8243
 200    8233
 100    8232
 70     8214
 Name: count, dtype: int64)

# 4.0 filtering data

In [6]:
sub_df = df[["ADate",	"MachineName", 	"Energy", "Device", "Gantry Angle", "Spot", "x-pos", "y-pos"]].copy()

## calculate abs shift

In [7]:
pred_xrv4000 = {'Top-Top-Left': [-125, -175], 'Top-Top-Centre': [0, -175], 'Top-Top-Right': [125, -175], \
                'Top-Left': [-125, -125], 'Top-Centre':[0, -125], 'Top-Right':[125, -125], \
                'Left': [-125, 0], 'Centre':[0, 0], 'Right':[125, 0], \
                'Bottom-Left': [-125, 125], 'Bottom-Centre':[0, 125], 'Bottom-Right':[125, 125], \
                'Bottom-Bottom-Left': [-125, 175], 'Bottom-Bottom-Centre': [0, 175], 'Bottom-Bottom-Right': [125, 175]}

sub_df['px_pos'] = sub_df['Spot'].map(lambda s: pred_xrv4000[s][0] if s in pred_xrv4000 else None)
sub_df['py_pos'] = sub_df['Spot'].map(lambda s: pred_xrv4000[s][1] if s in pred_xrv4000 else None)

sub_df['abs_xpos'] = sub_df["x-pos"] - sub_df["px_pos"]
sub_df['abs_ypos'] = sub_df["y-pos"] - sub_df["py_pos"]

In [8]:
print(sub_df.head(2))

                ADate MachineName  Energy    Device  Gantry Angle  \
0 2022-04-21 16:44:14    Gantry 2      70  XRV-3000           180   
1 2022-04-21 16:44:14    Gantry 2      70  XRV-3000           180   

            Spot     x-pos     y-pos  px_pos  py_pos  abs_xpos  abs_ypos  
0  Bottom-Centre   -0.2357  124.9224       0     125   -0.2357   -0.0776  
1    Bottom-Left -125.0153  125.4501    -125     125   -0.0153    0.4501  


In [9]:

def plotly_spot_position(df, pos, gantry, device, energy, gantry_angle, n_months):
    """ plot spot position time series data
        df = dataframe
        gantry = "Gantry 1", "Gantry 2", 
        pos = "abs_xpos",
        device = "XRV-3000", "XRV-4000"
        energy = int,
        gantry_angle = 0,90,180,270
        n_month = int

    
     """
    
     # only show data from last 12 months
    start_date = pd.Timestamp.today() - pd.DateOffset(months=n_months)

    selected_df = df[(df["MachineName"]==gantry) & (df["Device"] == device) & (df['ADate'] >= start_date) & (df["Energy"] == energy) &(df["Gantry Angle"] == gantry_angle)]

    # set colour
    palette = sns.color_palette("deep", n_colors=df['Spot'].nunique())
    palette_hex = [to_hex(c) for c in palette]


    # Plot
    fig = px.scatter(
        selected_df,
        x='ADate',
        y=pos,
        symbol='Spot', 
        color='Spot',        # hue
         color_discrete_sequence= px.colors.qualitative.T10,
        title=f'{gantry} - absolute shift- {pos}',
        labels={'x-pos': 'X Position', 'adate': 'Date'},
        height=500
    )

    # Add tolerance bands +/- 2
    fig.add_hline(y=2, line_dash="dash", line_color="grey", annotation_text="tolerance", annotation_position="top left")
    fig.add_hline(y=-2, line_dash="dash", line_color="grey")



    # Optional: connect points by spot for clarity
    fig.update_traces(mode='markers+lines',
                    marker=dict(size=12,               # larger size
                                line=dict(width=2)),     # outline width
                    line=dict(width=1                # thinner connecting lines
                    ))

    # Show plot
    fig.show()

    
    return 


# plotting absolute y-pos, on Gantry 4, from XRV-4000 data, 70 MeV spot, Gantry angle = 0, in last2 months
plotly_spot_position(sub_df, "abs_ypos", "Gantry 4", "XRV-4000", 70, 0, 24)


In [10]:
plotly_spot_position(sub_df, "abs_ypos", "Gantry 4", "XRV-4000", 70, 0, 24)

## plot another device

In [11]:
plotly_spot_position(sub_df, "abs_xpos", "Gantry 4", "XRV-3000", 70, 0, 24)

In [12]:
start_date = pd.Timestamp.today() - pd.DateOffset(months=12)
selected_df = sub_df[(df["MachineName"]=="Gantry 2") & (df["Device"] == "XRV-3000") & (df['ADate'] >= start_date)].copy()
# Calculate average abs_xpos per adate and energy
selected_df['avg_abs_pos'] = selected_df.groupby(['ADate', 'Energy', "Gantry Angle"])["abs_xpos"].transform('mean')

selected_df.head(5)

,ADate,MachineName,Energy,Device,Gantry Angle,Spot,x-pos,y-pos,px_pos,py_pos,abs_xpos,abs_ypos,avg_abs_pos
31254,2025-03-17 17:45:17,Gantry 2,70,XRV-3000,0,Bottom-Centre,0.0171,125.4348,0,125,0.0171,0.4348,0.081178
31255,2025-03-17 17:45:17,Gantry 2,70,XRV-3000,0,Bottom-Left,-125.1845,125.2629,-125,125,-0.1845,0.2629,0.081178
31256,2025-03-17 17:45:17,Gantry 2,70,XRV-3000,0,Bottom-Right,125.2672,125.4379,125,125,0.2672,0.4379,0.081178
31257,2025-03-17 17:45:17,Gantry 2,70,XRV-3000,0,Centre,0.0382,0.3277,0,0,0.0382,0.3277,0.081178
31258,2025-03-17 17:45:17,Gantry 2,70,XRV-3000,0,Left,-125.2104,0.6147,-125,0,-0.2104,0.6147,0.081178


In [13]:

def plotly_ave_spot_position(df, parameter, gantry, device,  n_months):
    """ plot average spot position across all spot positions with the same adate and energy
        df = dataframe
        gantry = "Gantry 1", "Gantry 2", 
        paramter = "abs_xpos",
        device = "XRV-3000", "XRV-4000"
        energy = int
        n_month = int

    
     """
    
     # only show data from last 12 months
    start_date = pd.Timestamp.today() - pd.DateOffset(months=n_months)

    selected_df = df[(df["MachineName"]==gantry) & (df["Device"] == device) & (df['ADate'] >= start_date)].copy()


    # Calculate average abs_xpos per adate and energy
    selected_df['avg_abs_pos'] = selected_df.groupby(['ADate', 'Energy', "Gantry Angle"])[parameter].transform('mean')

    # want to displace energy as discrete colour not spectrum
    selected_df['Energy'] = df['Energy'].astype(int).astype(str)

    

    # Plot
    fig = px.scatter(
        selected_df,
        x='ADate',
        y='avg_abs_pos',
        symbol='Gantry Angle', 
        color='Energy',        # hue
        title=f'average {parameter} across all spot positions with the same adate and energy',
        labels={'x-pos': 'X Position', 'adate': 'Date'},
        height=500
    )

    # Add tolerance bands +/- 2
    fig.add_hline(y=2, line_dash="dash", line_color="grey", annotation_text="tolerance", annotation_position="top left")
    fig.add_hline(y=-2, line_dash="dash", line_color="grey")



    # Optional: connect points by spot for clarity
       # Optional: connect points by spot for clarity
    fig.update_traces(mode='markers+lines',
                    marker=dict(size=12,               # larger size
                                line=dict(width=2)),     # outline width
                    line=dict(width=1                # thinner connecting lines
                    ))

    # Show plot
    fig.show()

    
    return 




In [14]:
plotly_ave_spot_position(sub_df, "abs_xpos", "Gantry 1", "XRV-3000",  24)

## FWHM

In [29]:
ref_df = pd.read_excel("/workspaces/sg-data-vis/data/xlsx_exported_from_access/ref/Ref_RS0_Dist0.xlsx")

print(ref_df.columns)
print(ref_df.head())

Index(['source', 'gantry', 'rs', 'dist', 'energy', 'x_stddev', 'y_stddev'], dtype='object')
  source  gantry  rs  dist  energy  x_stddev  y_stddev
0     G3     270   0     0      70  5.913573  6.000366
1     G3     270   0     0      75  5.808925  5.796172
2     G3     270   0     0      80  5.605678  5.515724
3     G3     270   0     0      85  5.494827  5.322688
4     G3     270   0     0      90  5.389959  5.203662


In [30]:
fwhm_cols = ["hor_fwhm", "vert_fwhm", "bltr_fwhm", "tlbr_fwhm"]

fwhm_df = df[[
    "ADate", "MachineName", "Energy", "Device", "Gantry Angle", "Spot", *fwhm_cols
]].copy()

FWHM_FACTOR = 2.3548200450309493

In [ ]:
def plotly_fwhm_time_series(
    df,
    gantry,
    device,
    energy,
    gantry_angle,
    n_months=12,
    fwhm_col="hor_fwhm",   # "hor_fwhm" / "vert_fwhm" / "bltr_fwhm" / "tlbr_fwhm" / "all"
    agg=None,              # None / "daily_median" / "daily_mean"
    ref_df=None,           # ref table (must have: source, gantry, energy, x_stddev, y_stddev)
    ref_source="TPS",      # "TPS"/"G1"/"G2"/...
    tol_frac=0.10
):
    start_date = pd.Timestamp.today() - pd.DateOffset(months=n_months)

    sel = df[
        (df["MachineName"] == gantry) &
        (df["Device"] == device) &
        (df["Energy"] == energy) &
        (df["Gantry Angle"] == gantry_angle) &
        (df["ADate"] >= start_date)
    ].copy()

    if sel.empty:
        print("No data after filtering.")
        return

    if agg in {"daily_median", "daily_mean"}:
        sel["Day"] = sel["ADate"].dt.floor("D")
        func = "median" if agg == "daily_median" else "mean"
        sel = sel.groupby(["Day", "Spot"], as_index=False)[fwhm_cols].agg(func)
        sel = sel.rename(columns={"Day": "ADate"})

    color_seq = px.colors.qualitative.T10

    # ---- build plot ----
    if fwhm_col == "all":
        long_df = sel.melt(
            id_vars=["ADate", "Spot"],
            value_vars=fwhm_cols,
            var_name="Direction",
            value_name="FWHM"
        )
        long_df["Direction"] = pd.Categorical(long_df["Direction"], categories=fwhm_cols, ordered=True)

        fig = px.scatter(
            long_df,
            x="ADate",
            y="FWHM",
            color="Spot",
            symbol="Spot",
            facet_col="Direction",
            facet_col_wrap=2,
            category_orders={"Direction": fwhm_cols},
            color_discrete_sequence=color_seq,
            title=f"{gantry} {device} {energy}MeV GA={gantry_angle} — FWHM (all directions)",
            height=700
        )
        fig.update_yaxes(matches=None)
    else:
        if fwhm_col not in fwhm_cols:
            raise ValueError(f"fwhm_col must be one of {fwhm_cols} or 'all'")

        fig = px.scatter(
            sel,
            x="ADate",
            y=fwhm_col,
            color="Spot",
            symbol="Spot",
            color_discrete_sequence=color_seq,
            title=f"{gantry} {device} {energy}MeV GA={gantry_angle} — {fwhm_col}",
            height=550
        )

# ---- add tol from ref ----
    if str(ref_source).strip().upper() == "TPS":
        ref_src = "G1"
        ref_ga = 90
    else:
        ref_src = ref_source
        ref_ga = gantry_angle

    ref = ref_df[
        (ref_df["source"].astype(str).str.strip().str.upper() == str(ref_src).strip().upper()) &
        (pd.to_numeric(ref_df["gantry"], errors="coerce") == ref_ga) &
        (pd.to_numeric(ref_df["energy"], errors="coerce") == energy)
    ]

    r = ref.iloc[0]
    x_sigma = float(r["x_stddev"])
    y_sigma = float(r["y_stddev"])

    hor_center = x_sigma * FWHM_FACTOR
    vert_center = y_sigma * FWHM_FACTOR

    def add_tol(center, row, col):
        y_hi = center * (1 + tol_frac)
        y_lo = center * (1 - tol_frac)

        fig.add_hline(y=y_hi, line_dash="dash", line_color="grey", line_width=2, row=row, col=col)
        fig.add_hline(y=y_lo, line_dash="dash", line_color="grey", line_width=2, row=row, col=col)

    if fwhm_col == "hor_fwhm":
        add_tol(hor_center, row=1, col=1)
    elif fwhm_col == "vert_fwhm":
        add_tol(vert_center, row=1, col=1)
    elif fwhm_col == "all":
        add_tol(hor_center, row=2, col=1)  # hor (top-left)
        add_tol(vert_center, row=2, col=2) # vert (top-right)

    # print tol used
    print(f"Using tol from source={ref_src}, GA={ref_ga}, energy={energy}MeV")

    fig.update_traces(
        mode="markers+lines",
        marker=dict(size=10, line=dict(width=1)),
        line=dict(width=1)
    )
    fig.show()

In [35]:
plotly_fwhm_time_series(
    fwhm_df,
    gantry="Gantry 4",
    device="XRV-4000",
    energy=100,
    gantry_angle=0,
    n_months=24,
    fwhm_col="all",
    agg="daily_median",
    ref_df=ref_df,
    ref_source="TPS",
    tol_frac=0.10
)


Using tol from source=G1, GA=90, energy=100MeV
